# Órdenes por sujeto y lugar

Usa el kernel `.venv-decision`. La variante aprende a distinguir la petición de movimiento de la charla. Extrae los indicativos y destinos del catálogo activo y admite una segunda clasificación opcional de postura. El entrenamiento usa CPU y guarda sus pesos dentro de esta variante.

In [1]:
from pathlib import Path
import sys
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'AI NLP/yell').is_dir())
variant = root / 'Variantes/SujetoLugar'
sys.path[:0] = [str(variant), str(root / 'AI NLP')]
from slots import SlotClassifier, fit
from yell.decision_data import read_data
{k: len(rows) for k, rows in read_data().items()}

{'train': 1318, 'val': 298, 'test': 314}

## Entrenamiento y validación

Las frases de cada familia se asignan a una sola partición. Los ejemplos adicionales están en `frases.json`. El clasificador usa palabras y fragmentos de caracteres, regularización L2 y selección de umbral en validación. La penalización prioriza evitar movimientos durante una charla.

In [2]:
fit()

{'frases_normalizadas': 395, 'umbral_validacion': 0.5}

## Evaluación

La comparación mide el mismo par de campos para ambos clasificadores. Las órdenes médicas y de cancelación quedan fuera del vocabulario de movimiento. La postura tiene una métrica adicional. La prueba de voz se ejecuta por separado con `comparar.py`.

In [3]:
from comparar import test_text
from yell.decision import DecisionModel
informe = test_text(DecisionModel(), SlotClassifier())
{k: {m: n for m, n in v.items() if m != 'errores'} for k, v in informe.items()}

c:\Users\aaran\Documents\Semestre 2-26\SIS421\SIS421_02-2026\NLP\Proyecto final\.venv-decision\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'actual': {'n': 314,
  'exactos': 288,
  'ordenes_esperadas': 220,
  'ordenes_recuperadas': 194,
  'sujeto_o_destino_erroneo': 0,
  'movimientos_indebidos': 0,
  'p50_ms': 5.52,
  'p95_ms': 9.17},
 'sujeto_lugar': {'n': 314,
  'exactos': 300,
  'ordenes_esperadas': 220,
  'ordenes_recuperadas': 210,
  'sujeto_o_destino_erroneo': 0,
  'movimientos_indebidos': 4,
  'p50_ms': 1.72,
  'p95_ms': 2.65},
 'sujeto_lugar_postura': {'n': 314,
  'exactos': 300,
  'ordenes_esperadas': 220,
  'ordenes_recuperadas': 210,
  'sujeto_o_destino_erroneo': 0,
  'movimientos_indebidos': 4,
  'p50_ms': 2.5,
  'p95_ms': 3.87,
  'exactos_incluida_postura': 269}}

## Probar una frase

Cambia la frase, el pelotón, los nombres activos o la ubicación de «acá». Activa `posture=True` para probar avanzar y mantener.

In [13]:
from yell.contract import UNITS
modelo = SlotClassifier(posture=False)
roster = {'Alfa-1': {'nombre': 'Alex'}, 'Alfa-2': {'nombre': 'Juan'}}
aliases = {'Alfa': UNITS['Alfa'], 'Bravo': UNITS['Bravo'], **{k:[v['nombre']] for k,v in roster.items()}}
frase = 'Medico vayan a ayudar al flanco derecho'
decision, probabilidades, ruta = modelo.classify(None, frase, aliases, roster, 'centro', False)
print(decision, probabilidades)

{'tipo': 'charla', 'interlocutor': None} [('charla', 1.0)]
